# sjkx 竞品对照分析（JupyterLab + Pandas）

连接 Neon，查看各表 `snapshot_date` 分布，并运行 `sql/` 下的对照查询。

**启动**（项目根目录）：

```bash
source .venv/bin/activate
jupyter lab
```

配置来自 `database.env`（由 `load_config()` 加载，不要用 `source database.env`）。

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analysis import SQL_DIR, fetch_snapshot_summary, load_sql
from src.config import load_config
from src.db import get_connection

load_config()
pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("项目根:", ROOT)

In [ ]:
conn = get_connection()
try:
    summary = fetch_snapshot_summary(conn, limit=15)
finally:
    conn.close()

summary

In [ ]:
# 修改锚定日（须存在于对应表；两两对照用 F 的日期，三方参照用 V 的日期）
ANCHOR_F = "2026-05-20"  # shop_comparison：F 锚定
ANCHOR_V = "2026-05-21"  # vestiaire_reference：V 锚定

latest = summary.groupby("tbl")["snapshot_date"].max()
print("各表最新 snapshot_date:")
print(latest.to_string())

## 竞品两两对照（F 锚定 → R 最近邻）

输出按品牌 / 型号 / 成色的目录聚合指标（F 与 R 并排）。若 R 表日期与 F 相差超过 3 天，结果可能为空。

In [ ]:
from datetime import date

sql_shop = load_sql(SQL_DIR / "shop_comparison.sql", date.fromisoformat(ANCHOR_F))
conn = get_connection()
try:
    df_shop = pd.read_sql(sql_shop, conn)
finally:
    conn.close()

print(f"行数: {len(df_shop)}")
if len(df_shop):
    print(
        "配对:",
        df_shop[["f_date", "r_date", "pairing_gap_days"]].drop_duplicates().iloc[0].to_dict(),
    )
df_shop.head(20)

## 竞品三方参照（V 锚定 → F、R 最近邻）

In [ ]:
sql_vref = load_sql(
    SQL_DIR / "vestiaire_reference_comparison.sql",
    date.fromisoformat(ANCHOR_V),
)
conn = get_connection()
try:
    df_vref = pd.read_sql(sql_vref, conn)
finally:
    conn.close()

print(f"行数: {len(df_vref)}")
df_vref.head(20)

## 探索：按品牌汇总（示例）

在对照结果上继续做 Pandas 筛选 / 排序；V 表含 `currency` 时三方结果需按币种分组解读。

In [ ]:
if len(df_shop) == 0:
    print("两两对照无数据，请检查 ANCHOR_F 或补导 R 表接近日期的快照。")
else:
    by_brand = (
        df_shop.groupby("brand", dropna=False)
        .agg(
            f_items=("f_item_count", "sum"),
            r_items=("r_item_count", "sum"),
            f_avg=("f_avg_price", "mean"),
            r_avg=("r_avg_price", "mean"),
        )
        .sort_values("f_items", ascending=False)
    )
    by_brand.head(15)

In [ ]:
try:
    import matplotlib.pyplot as plt

    if len(df_shop) > 0:
        top = by_brand.head(10).dropna(how="all")
        fig, ax = plt.subplots(figsize=(10, 4))
        x = range(len(top))
        w = 0.35
        ax.bar([i - w / 2 for i in x], top["f_avg"], width=w, label="F 均价")
        ax.bar([i + w / 2 for i in x], top["r_avg"], width=w, label="R 均价")
        ax.set_xticks(list(x))
        ax.set_xticklabels(top.index, rotation=45, ha="right")
        ax.legend()
        ax.set_title(f"Top 品牌均价对比（锚定 F={ANCHOR_F}）")
        plt.tight_layout()
        plt.show()
except ImportError:
    print("未安装 matplotlib，跳过图表。可执行: pip install matplotlib")